# Large-Scale Adversarial Training for Vision-and-Language Representation Learning (VILLA)

# https://arxiv.org/pdf/2006.06195

---

## Abstract

This paper introduces **VILLA** (Vision-and-Language Large-scale Adversarial Training), the first
framework to apply large-scale adversarial training (AT) to vision-and-language (V+L)
representation learning. VILLA operates in two stages: task-agnostic adversarial pre-training (APT)
followed by task-specific adversarial fine-tuning (AFT). Adversarial perturbations are applied in
the embedding space of each modality rather than at the pixel or token level. The framework
employs a "free" adversarial training strategy combined with KL-divergence-based regularization
to improve model generalization. VILLA achieves state-of-the-art performance across six major
V+L benchmarks.

---

## Problems

- Large-scale pre-trained V+L models are prone to **overfitting** when fine-tuned on
  downstream tasks with limited labeled data.
- Conventional adversarial training operates at the **pixel or discrete token level**, which is
  rigid, computationally expensive, and difficult to apply to multimodal settings.
- Standard adversarial training in the image domain is known to **hurt model accuracy**
  on clean inputs — a trade-off between robustness and generalization.
- No prior work had explored adversarial training systematically for **both pre-training and
  fine-tuning** stages in V+L models.

---

## Proposed Solutions

- **Two-stage adversarial training**: Apply AT during both pre-training (APT) and fine-tuning
  (AFT) using the same mathematical formulation.
- **Embedding-space perturbations**: Add adversarial perturbations directly to word embeddings
  (text) and image-region features (image), avoiding the limitations of pixel/token-level attacks.
- **"Free" adversarial training strategy**: Accumulate parameter gradients across multiple PGD
  steps without extra forward-backward passes, enabling efficient large-scale training.
- **KL-divergence regularization**: Enforce consistency between clean and adversarial prediction
  distributions to promote smoother, more generalizable representations.

---

## Purpose

The primary purpose of VILLA is to improve the **generalization ability** of pre-trained V+L
models by leveraging adversarial training as a regularization mechanism — not to craft
interpretable adversarial examples — thereby achieving consistent performance gains across
diverse downstream V+L tasks in both pre-training and fine-tuning stages.

---

## Methodology

### Training Objective

The unified training objective minimizes:

$$\min_{\theta} \mathbb{E}_{(x_{\text{img}}, x_{\text{txt}}, y) \sim D}
\left[ \mathcal{L}_{\text{std}}(\theta) + R_{\text{at}}(\theta) + \alpha \cdot R_{\text{kl}}(\theta) \right]$$

where:
- $\mathcal{L}_{\text{std}}(\theta)$ is the standard cross-entropy loss on clean data.
- $R_{\text{at}}(\theta)$ is the label-preserving adversarial loss:

$$R_{\text{at}}(\theta) = \max_{\|\delta_{\text{img}}\| \leq \epsilon} \mathcal{L}(f_\theta(x_{\text{img}} + \delta_{\text{img}}, x_{\text{txt}}), y)
+ \max_{\|\delta_{\text{txt}}\| \leq \epsilon} \mathcal{L}(f_\theta(x_{\text{img}}, x_{\text{txt}} + \delta_{\text{txt}}), y)$$

- $R_{\text{kl}}(\theta)$ is the KL-divergence regularization term:

$$R_{\text{kl}}(\theta) = \max_{\|\delta_{\text{img}}\| \leq \epsilon}
\mathcal{L}_{\text{kl}}\left(f_\theta(x_{\text{img}} + \delta_{\text{img}}, x_{\text{txt}}),\, f_\theta(x_{\text{img}}, x_{\text{txt}})\right)
+ \max_{\|\delta_{\text{txt}}\| \leq \epsilon}
\mathcal{L}_{\text{kl}}\left(f_\theta(x_{\text{img}}, x_{\text{txt}} + \delta_{\text{txt}}),\, f_\theta(x_{\text{img}}, x_{\text{txt}})\right)$$

where $\mathcal{L}_{\text{kl}}(p, q) = \text{KL}(p \| q) + \text{KL}(q \| p)$.

### Perturbation Update (PGD Step)

$$\delta_{\text{img},\, t+1} = \Pi_{\|\delta_{\text{img}}\| \leq \epsilon}
\left( \delta_{\text{img},\, t} + \alpha \cdot \frac{g(\delta_{\text{img},\, t})}{\|g(\delta_{\text{img},\, t})\|_F} \right)$$

### "Free" Adversarial Training

- Perform $K$ PGD iterations per mini-batch.
- Accumulate parameter gradients $\nabla_\theta \mathcal{L}$ across all iterations at no extra cost.
- Update model parameters once using the accumulated gradients — effectively a
  $K$-times-larger virtual mini-batch.
- Perturbations for image and text modalities are updated **alternately**, not simultaneously.

### Models and Datasets

| Component | Details |
|---|---|
| Primary backbone | UNITER-base (12L, 768d) and UNITER-large (24L, 1024d) |
| Secondary backbone | LXMERT (two-stream architecture) |
| Pre-training data | COCO, Visual Genome, Conceptual Captions, SBU Captions |
| Image features | Bottom-up region features (Anderson et al., CVPR 2018) |
| Pre-training tasks | Masked Language Modeling (MLM), Image-Text Matching (ITM) |
| Adversarial steps $K$ | 3 |

### Downstream Tasks and Benchmarks

| Task | Dataset/Benchmark |
|---|---|
| Visual Question Answering | VQA v2 |
| Visual Commonsense Reasoning | VCR |
| Natural Language Visual Reasoning | NLVR2 |
| Visual Entailment | SNLI-VE |
| Referring Expression Comprehension | RefCOCO, RefCOCO+, RefCOCOg |
| Image-Text Retrieval | Flickr30k (IR and TR) |

---

## Results

### Main Results vs. State-of-the-Art (UNITER backbone)

| Model | VQA (test-std) | VCR Q→AR | NLVR2 (test-P) | SNLI-VE (test) | Flickr30k IR R@1 |
|---|---|---|---|---|---|
| UNITER-base | 72.91 | 58.2 | 77.85 | 78.28 | 72.52 |
| VILLA-base | 73.67 | 60.6 | 79.30 | 79.03 | 74.74 |
| UNITER-large | 74.02 | 62.8 | 79.98 | 79.38 | 75.56 |
| VILLA-large | **74.87** | **65.7** | **81.47** | **80.02** | **76.26** |
| VILLA-large (Ensemble) | **75.85** | — | — | — | — |

### Ablation: Pre-training vs. Fine-tuning (UNITER-base)

| Method | Average Score |
|---|---|
| UNITER (reimp.) | 78.06 |
| VILLA-pre only | 78.57 (+0.51) |
| VILLA-fine only | 78.88 (+0.82) |
| VILLA (both) | **79.21 (+1.15)** |

### Key Findings

- Adversarial perturbations on **image features alone** boost clean-data performance,
  contradicting the conventional robustness-accuracy trade-off.
- Adding perturbations to **one modality at a time** is sufficient; simultaneous perturbation
  yields marginal additional benefit.
- VILLA consistently outperforms **FreeLB** (text-only AT baseline) across all tasks.
- **Probing analysis** shows VILLA achieves higher attention weights for visual coreference
  and visual relation tasks (0.223 vs. 0.195 average), indicating stronger multimodal alignment.
- Applied to **LXMERT**, adversarial fine-tuning yields +0.88 average gain across VQA, GQA,
  and NLVR2, demonstrating framework generalizability.
- VILLA exhibits clear **regularization behavior**: higher validation accuracy, lower training
  accuracy compared to the baseline.

---

## Conclusions

VILLA establishes adversarial training as a principled and effective regularization strategy for
vision-and-language representation learning. By performing AT in the embedding space across
both pre-training and fine-tuning stages, and by introducing KL-divergence-based adversarial
regularization on top of a computationally efficient "free" training strategy, VILLA achieves
consistent and significant performance improvements across all evaluated V+L benchmarks.
The framework is model-agnostic and generalizes across different V+L architectures. Future
work is directed toward accelerating the adversarial pre-training procedure to reduce
computational overhead, and toward a more systematic investigation of adversarial robustness
in multimodal settings.

# Mathematical & Statistical Content: VILLA

---

## 1. Core Training Paradigm — Empirical Risk Minimization (ERM)

### Equation

$$\min_{\theta} \mathbb{E}_{(x_{\text{img}}, x_{\text{txt}}, y) \sim D}
\left[ \mathcal{L}(f_\theta(x_{\text{img}}, x_{\text{txt}}), y) \right]$$

### Explanation

This is the **standard supervised learning objective**. The model $f_\theta$ takes an image
$x_{\text{img}}$ and text $x_{\text{txt}}$ as input and predicts output $y$. The goal is to find
parameters $\theta$ that minimize the **expected loss** (cross-entropy) over the training
distribution $D$. Both pre-training and fine-tuning share this same formulation — only the
definition of $y$ changes (e.g., masked token, binary match label, VQA answer).

---

## 2. Full VILLA Training Objective

### Equation

$$\min_{\theta} \mathbb{E}_{(x_{\text{img}}, x_{\text{txt}}, y) \sim D}
\left[ \mathcal{L}_{\text{std}}(\theta) + R_{\text{at}}(\theta) + \alpha \cdot R_{\text{kl}}(\theta) \right]$$

### Explanation

The total loss has **three components**:

| Term | Name | Role |
|---|---|---|
| $\mathcal{L}_{\text{std}}(\theta)$ | Standard cross-entropy loss | Learn from clean (unperturbed) data |
| $R_{\text{at}}(\theta)$ | Adversarial training loss | Learn to be robust to worst-case perturbations |
| $\alpha \cdot R_{\text{kl}}(\theta)$ | KL regularization loss | Enforce prediction confidence consistency |

$\alpha$ is a scalar **weighting hyperparameter** that controls the strength of KL
regularization (selected from $\{1.0, 1.5, 2.0\}$ in experiments).

---

## 3. Label-Preserving Adversarial Loss — $R_{\text{at}}(\theta)$

### Equation

$$R_{\text{at}}(\theta) =
\max_{\|\delta_{\text{img}}\|_F \leq \epsilon} \mathcal{L}(f_\theta(x_{\text{img}} + \delta_{\text{img}},\, x_{\text{txt}}),\, y)
+
\max_{\|\delta_{\text{txt}}\|_F \leq \epsilon} \mathcal{L}(f_\theta(x_{\text{img}},\, x_{\text{txt}} + \delta_{\text{txt}}),\, y)$$

### Explanation

This term finds the **worst-case perturbation** for each modality separately:

- $\delta_{\text{img}}$ is a small additive noise injected into **image region features**
- $\delta_{\text{txt}}$ is a small additive noise injected into **word embeddings**
- Both are constrained by a **Frobenius norm ball** of radius $\epsilon$, keeping
  perturbations small enough to preserve the original label $y$
- The **inner maximization** finds the perturbation that maximally increases the loss
- The **outer minimization** (in the full objective) trains $\theta$ to be robust to it
- This is a **minimax problem** — a classic formulation in adversarial machine learning

The two modalities are perturbed **one at a time**, not simultaneously.

---

## 4. Frobenius Norm Constraint

### Definition

$$\|\delta\|_F = \sqrt{\sum_{i,j} \delta_{ij}^2}$$

### Explanation

The **Frobenius norm** measures the overall magnitude of a matrix (or vector) of
perturbations. Constraining $\|\delta\|_F \leq \epsilon$ ensures the adversarial noise is
**small enough** not to change the semantic meaning of the input — it is the
multimodal analogue of an $\ell_2$ ball constraint commonly used in adversarial ML.

---

## 5. PGD — Projected Gradient Descent (Inner Maximization Solver)

### Equation

$$\delta_{\text{img},\, t+1} =
\Pi_{\|\delta_{\text{img}}\|_F \leq \epsilon}
\left(
\delta_{\text{img},\, t} + \alpha_{\text{adv}} \cdot
\frac{g(\delta_{\text{img},\, t})}{\|g(\delta_{\text{img},\, t})\|_F}
\right)$$

where $g(\delta_{\text{img},\, t}) = \nabla_{\delta_{\text{img}}} \mathcal{L}(f_\theta(x_{\text{img}} + \delta_{\text{img}},\, x_{\text{txt}}),\, y)$

### Explanation

PGD is used to **approximately solve the inner maximization** (find the worst perturbation):

- At each step $t$, compute the gradient of the loss with respect to $\delta_{\text{img}}$
- Take a **normalized gradient ascent step** of size $\alpha_{\text{adv}}$ to increase the loss
- **Project** the result back onto the $\epsilon$-ball using $\Pi$ to maintain the norm constraint
- Repeat for $K = 3$ steps (chosen as the number of adversarial training steps)
- The normalization $g / \|g\|_F$ makes step sizes consistent regardless of gradient magnitude

The same procedure applies symmetrically to $\delta_{\text{txt}}$.

---

## 6. KL-Divergence Regularization — $R_{\text{kl}}(\theta)$

### Equation

$$R_{\text{kl}}(\theta) =
\max_{\|\delta_{\text{img}}\|_F \leq \epsilon}
\mathcal{L}_{\text{kl}}\!\left(f_\theta(x_{\text{img}} + \delta_{\text{img}}, x_{\text{txt}}),\; f_\theta(x_{\text{img}}, x_{\text{txt}})\right)$$
$$+\;
\max_{\|\delta_{\text{txt}}\|_F \leq \epsilon}
\mathcal{L}_{\text{kl}}\!\left(f_\theta(x_{\text{img}}, x_{\text{txt}} + \delta_{\text{txt}}),\; f_\theta(x_{\text{img}}, x_{\text{txt}})\right)$$

where:

$$\mathcal{L}_{\text{kl}}(p, q) = \text{KL}(p \| q) + \text{KL}(q \| p)$$

and the **KL divergence** is:

$$\text{KL}(p \| q) = \sum_{i} p_i \log \frac{p_i}{q_i}$$

### Explanation

- $p$ is the model's output **probability distribution under adversarial input**
- $q$ is the model's output distribution under **clean input** (used as a soft target)
- $\mathcal{L}_{\text{kl}}$ is a **symmetric KL divergence** (also called Jeffrey divergence)
  measuring how different the two distributions are
- The goal: even under worst-case perturbation, the model's **confidence profile** should
  remain close to what it produces on clean data
- This goes beyond label-preservation — it constrains the **full output distribution**,
  not just the argmax prediction
- This captures **"dark knowledge"** (Hinton's term) — the relative probabilities
  across all classes, not just the top-1 label
- Mathematically, this promotes **Lipschitz smoothness** of $f_\theta$ around clean inputs

---

## 7. "Free" Adversarial Training — Gradient Accumulation

### Procedure (Algorithm 1)

$$g_t \leftarrow g_{t-1} + \frac{1}{K}
\mathbb{E}_{B}\left[ \nabla_\theta \left(\mathcal{L}_{\text{std}} + R_{\text{at}} + R_{\text{kl}}\right) \right]$$

$$\theta \leftarrow \theta - \tau \cdot g_K$$

### Explanation

Standard $K$-step PGD requires $K$ full forward-backward passes, making it
$K$ times more expensive than standard training. The **"free" strategy** avoids this:

- During each PGD step, the gradient with respect to $\theta$ is computed **for free**
  as a byproduct of computing the gradient w.r.t. $\delta$
- These parameter gradients are **accumulated** across all $K$ steps
- A **single parameter update** is performed at the end using the accumulated gradient $g_K$
- This effectively trains on a **$K$-times-larger virtual mini-batch** at the cost of one
  standard training step
- Perturbations $\delta_{\text{img}}$ and $\delta_{\text{txt}}$ are initialized from a
  uniform distribution: $\delta_0 \sim \mathcal{U}(-\epsilon, \epsilon)$, then updated
  via normalized gradient ascent at each step

---

## 8. Perturbation Initialization

### Equation

$$\delta_0 \leftarrow \frac{1}{\sqrt{N_\delta}} \cdot \mathcal{U}(-\epsilon, \epsilon)$$

### Explanation

- $N_\delta$ is the dimensionality of the perturbation vector
- The $1/\sqrt{N_\delta}$ scaling **normalizes the initial perturbation magnitude**
  so that it roughly respects the $\epsilon$-ball constraint from the start
- This is a standard initialization strategy in PGD-based adversarial training

---

## 9. Projection Operator $\Pi$

### Definition

$$\Pi_{\|\delta\|_F \leq \epsilon}(\delta') =
\begin{cases}
\delta' & \text{if } \|\delta'\|_F \leq \epsilon \\
\epsilon \cdot \dfrac{\delta'}{\|\delta'\|_F} & \text{otherwise}
\end{cases}$$

### Explanation

After each gradient ascent step, the updated perturbation may **violate the norm constraint**.
The projection $\Pi$ simply **rescales** the perturbation back onto the surface of the
$\epsilon$-ball if it exceeds it. This is the standard Euclidean projection onto an
$\ell_2$ ball and ensures the adversarial noise always remains bounded.

---

## 10. Statistical Evaluation — Attention Head Probing

### Metric

$$\text{Score}_{\text{head}} = \max_{j \in \text{phrase}} A_{r,\, j}$$

where $A_{r,j}$ is the attention weight from image region $r$ to text token $j$.

### Explanation

- To measure how well the model captures **visual coreference** and **visual relations**,
  the authors extract attention weights from all 144 attention heads
- For each noun phrase (potentially multi-token), the **maximum attention weight**
  between the image region and any token in the phrase is taken as the alignment score
- Scores are compared between UNITER and VILLA across 10 semantic categories
- VILLA achieves a higher average score (**0.223 vs. 0.195**), indicating stronger and
  more accurate multimodal alignment learned through adversarial training

---

## Summary Table of All Mathematical Components

| Concept | Mathematical Form | Purpose |
|---|---|---|
| ERM objective | $\min_\theta \mathbb{E}[\mathcal{L}]$ | Base supervised learning |
| Full VILLA loss | $\mathcal{L}_{\text{std}} + R_{\text{at}} + \alpha R_{\text{kl}}$ | Combined training objective |
| Adversarial loss $R_{\text{at}}$ | $\max_{\|\delta\|_F \leq \epsilon} \mathcal{L}(f_\theta(x+\delta), y)$ | Worst-case robustness |
| KL regularization $R_{\text{kl}}$ | $\text{KL}(p\|q) + \text{KL}(q\|p)$ | Distribution-level smoothness |
| PGD update | $\delta_{t+1} = \Pi(\delta_t + \alpha \cdot g/\|g\|_F)$ | Inner maximization solver |
| Frobenius norm | $\|\delta\|_F \leq \epsilon$ | Perturbation size constraint |
| Projection $\Pi$ | Rescale to $\epsilon$-ball | Enforce norm constraint |
| Free gradient accumulation | $g_t \leftarrow g_{t-1} + \frac{1}{K}\nabla_\theta \mathcal{L}$ | Efficient large-scale AT |
| Attention probing score | $\max_{j} A_{r,j}$ | Multimodal alignment measurement |

In [1]:
# ============================================================
# VILLA: Vision-and-Language Large-scale Adversarial Training
# Simplified Educational Implementation on CIFAR-10
# ============================================================
# This notebook replicates the core ideas of VILLA:
#   1. Embedding-space adversarial perturbations (image features)
#   2. "Free" adversarial training (gradient accumulation)
#   3. KL-divergence regularization (distribution consistency)
#   4. Two-stage: standard pre-warm + adversarial fine-tuning
# We adapt the multimodal framework to a single-modality
# image classification setting on CIFAR-10 for clarity.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── Device ───────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# ============================================================
# SECTION 1 — DATASET PREPARATION
# We use a 5 000-sample CIFAR-10 subset for fast training.
# CIFAR-10 has 10 classes of 32×32 colour images.
# ============================================================

CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]
NUM_CLASSES   = 10
SUBSET_TRAIN  = 5000   # training samples
SUBSET_VAL    = 1000   # validation samples
BATCH_SIZE    = 64
NUM_EPOCHS    = 5
ADV_STEPS     = 3      # K PGD steps (same as paper)
EPS           = 0.03   # perturbation radius ε
ADV_LR        = 1e-2   # adversarial step size α
KL_WEIGHT     = 1.5    # α coefficient for R_kl  (from paper)
WARMUP_EPOCHS = 2      # standard training before adversarial AT

# --- Transforms ---
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

# --- Load full datasets ---
full_train = torchvision.datasets.CIFAR10(
    root='./data', train=True,  download=True, transform=train_transform)
full_val   = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=val_transform)

# --- Balanced subset: 500 per class for training ---
def balanced_subset(dataset, n_total, n_classes=10):
    per_class = n_total // n_classes
    indices, counts = [], [0] * n_classes
    for i, (_, label) in enumerate(dataset):
        if counts[label] < per_class:
            indices.append(i)
            counts[label] += 1
        if sum(counts) >= n_total:
            break
    return Subset(dataset, indices)

train_subset = balanced_subset(full_train, SUBSET_TRAIN)
val_subset   = balanced_subset(full_val,   SUBSET_VAL)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f"Training samples : {len(train_subset)}")
print(f"Validation samples: {len(val_subset)}")

  1%|          | 1.05M/170M [00:48<58:35, 48.2kB/s]  

In [ ]:
# ============================================================
# SECTION 2 — MODEL DEFINITION
# We build a lightweight CNN whose *penultimate layer* acts as
# the "embedding space" — the layer where VILLA injects
# adversarial perturbations (analogous to image-region features
# in the original paper).
# ============================================================

class VILLAEncoder(nn.Module):
    """
    CNN feature extractor.
    Outputs a flat embedding vector — this is the space
    where adversarial perturbations δ are added (like the
    bottom-up region features in the original VILLA paper).
    """
    def __init__(self, embed_dim=256):
        super().__init__()
        # Block 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),   # 32→16
            nn.Dropout2d(0.1),
        )
        # Block 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),   # 16→8
            nn.Dropout2d(0.1),
        )
        # Block 3
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2),   # 8→4
        )
        # Project to embedding
        self.pool    = nn.AdaptiveAvgPool2d(1)           # →(B,256,1,1)
        self.proj    = nn.Linear(256, embed_dim)
        self.embed_dim = embed_dim

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.pool(x).flatten(1)   # (B, 256)
        z = F.relu(self.proj(x))      # (B, embed_dim)  ← "embedding space"
        return z


class VILLAClassifier(nn.Module):
    """
    Full VILLA model:
      Encoder  →  embedding z  →  MLP head  →  class logits
    Adversarial perturbations are injected into z
    (the embedding), not into the raw pixels.
    """
    def __init__(self, embed_dim=256, num_classes=10):
        super().__init__()
        self.encoder = VILLAEncoder(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def encode(self, x):
        """Return embedding (used to inject perturbations)."""
        return self.encoder(x)

    def classify(self, z):
        """Classify from embedding."""
        return self.head(z)

    def forward(self, x):
        z = self.encode(x)
        return self.classify(z)


model = VILLAClassifier(embed_dim=256, num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

In [ ]:
# ============================================================
# SECTION 3 — LOSS FUNCTIONS
# VILLA uses three loss components (Eq. 2 in paper):
#   L_std  : cross-entropy on clean embeddings
#   R_at   : cross-entropy on adversarially perturbed embeddings
#   R_kl   : symmetric KL divergence between clean & adv distributions
# ============================================================

def cross_entropy_loss(logits, labels):
    """Standard classification cross-entropy loss (L_std)."""
    return F.cross_entropy(logits, labels)

def symmetric_kl_loss(p_logits, q_logits):
    """
    Symmetric KL divergence: KL(p||q) + KL(q||p)
    This is R_kl in the paper — enforces that the model's
    confidence profile does not change under perturbation.
    p_logits: adversarial output logits
    q_logits: clean output logits  (treated as soft target)
    """
    p = F.softmax(p_logits, dim=-1)
    q = F.softmax(q_logits, dim=-1)
    # Add small ε for numerical stability
    kl_pq = F.kl_div((q + 1e-8).log(), p + 1e-8, reduction='batchmean')
    kl_qp = F.kl_div((p + 1e-8).log(), q + 1e-8, reduction='batchmean')
    return kl_pq + kl_qp

In [ ]:
# ============================================================
# SECTION 4 — "FREE" ADVERSARIAL TRAINING ALGORITHM
# Implements Algorithm 1 from the VILLA paper.
#
# Key idea: instead of K separate forward-backward passes,
# we accumulate ∇_θ L during each PGD step (for free) and
# do a single parameter update at the end.
# Perturbations are in EMBEDDING SPACE (not pixel space).
# ============================================================

def free_adversarial_training_step(model, images, labels,
                                   optimizer, eps, adv_lr,
                                   adv_steps, kl_weight):
    """
    VILLA "Free" Multi-modal Adversarial Training Step.

    Steps:
    1. Encode images to get clean embeddings z.
    2. Initialise perturbation δ ~ Uniform(-ε, ε).
    3. For K steps:
         a. Compute clean logits (soft target for KL).
         b. Compute adversarial logits (z + δ).
         c. Compute L_std + R_at + α·R_kl.
         d. Accumulate ∇_θ L  (free gradient).
         e. Gradient-ascend δ (normalised step).
         f. Project δ back onto ε-ball.
    4. Update θ once with accumulated gradients.

    Returns individual loss components for logging.
    """
    model.train()
    optimizer.zero_grad()

    # --- Encode images to embedding space (no grad needed for encoder
    #     during perturbation — we freeze the encoder momentarily) ---
    with torch.no_grad():
        z_clean = model.encode(images)   # (B, embed_dim)

    # --- Initialise perturbation δ uniformly in [-ε, ε] ---
    delta = torch.empty_like(z_clean).uniform_(-eps, eps)
    delta.requires_grad_(True)

    # Accumulated parameter gradients across K steps
    accumulated_param_grads = [torch.zeros_like(p)
                                for p in model.parameters()]

    loss_std_total = 0.0
    loss_at_total  = 0.0
    loss_kl_total  = 0.0

    for step in range(adv_steps):
        # Zero gradients for a fresh accumulation each sub-step
        optimizer.zero_grad()

        # ---- Clean logits (soft target for KL) ----
        logits_clean = model.classify(z_clean)          # (B, C)

        # ---- Adversarial logits (perturbed embedding) ----
        z_adv    = z_clean + delta                       # inject δ
        logits_adv = model.classify(z_adv)              # (B, C)

        # ---- L_std: cross-entropy on CLEAN embeddings ----
        l_std = cross_entropy_loss(logits_clean, labels)

        # ---- R_at: cross-entropy on ADVERSARIAL embeddings ----
        l_at = cross_entropy_loss(logits_adv, labels)

        # ---- R_kl: symmetric KL divergence ----
        l_kl = symmetric_kl_loss(logits_adv, logits_clean.detach())

        # ---- Total loss ----
        total_loss = l_std + l_at + kl_weight * l_kl

        # ---- Backward to get ∇_θ L  (free accumulation) ----
        total_loss.backward(retain_graph=True)

        # Accumulate parameter gradients (1/K weighting)
        with torch.no_grad():
            for i, p in enumerate(model.parameters()):
                if p.grad is not None:
                    accumulated_param_grads[i] += p.grad.clone() / adv_steps

        # ---- Gradient ascent on δ  (maximise loss) ----
        if delta.grad is not None:
            delta.grad.zero_()

        # Re-compute gradient w.r.t δ only
        optimizer.zero_grad()
        z_adv2      = z_clean + delta
        logits_adv2 = model.classify(z_adv2)
        loss_for_delta = (cross_entropy_loss(logits_adv2, labels) +
                          kl_weight * symmetric_kl_loss(
                              logits_adv2, logits_clean.detach()))
        loss_for_delta.backward()

        with torch.no_grad():
            # Normalised gradient ascent step
            g = delta.grad
            if g is not None and g.norm() > 0:
                delta_new = delta + adv_lr * g / (g.norm(dim=-1, keepdim=True) + 1e-8)
                # Project back onto ε-ball (Frobenius-norm constraint)
                norms = delta_new.norm(dim=-1, keepdim=True)
                delta_new = torch.where(
                    norms > eps,
                    eps * delta_new / (norms + 1e-8),
                    delta_new
                )
                delta.data.copy_(delta_new)

        loss_std_total += l_std.item()
        loss_at_total  += l_at.item()
        loss_kl_total  += l_kl.item()

    # ---- Apply accumulated parameter gradients once ----
    optimizer.zero_grad()
    with torch.no_grad():
        for i, p in enumerate(model.parameters()):
            if p.grad is not None:
                p.grad.copy_(accumulated_param_grads[i])
            else:
                p.grad = accumulated_param_grads[i].clone()
    optimizer.step()

    return (loss_std_total / adv_steps,
            loss_at_total  / adv_steps,
            loss_kl_total  / adv_steps)


def standard_training_step(model, images, labels, optimizer):
    """
    Standard (non-adversarial) training step.
    Used during the warm-up phase before AT is activated
    (mirrors VILLA's 'standard pre-training then adversarial' schedule).
    """
    model.train()
    optimizer.zero_grad()
    logits = model(images)
    loss   = cross_entropy_loss(logits, labels)
    loss.backward()
    optimizer.step()
    return loss.item(), 0.0, 0.0   # R_at and R_kl are zero in warm-up

In [ ]:
# ============================================================
# SECTION 5 — VALIDATION / EVALUATION
# ============================================================

def evaluate(model, loader):
    """
    Evaluate on a dataloader.
    Returns accuracy, average loss, per-class accuracy,
    all predicted labels, all true labels, and all confidences.
    """
    model.eval()
    correct, total = 0, 0
    total_loss = 0.0
    all_preds, all_labels, all_confs = [], [], []
    class_correct = [0] * NUM_CLASSES
    class_total   = [0] * NUM_CLASSES

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss   = cross_entropy_loss(logits, labels)
            total_loss += loss.item()

            probs      = F.softmax(logits, dim=-1)
            conf, preds = probs.max(dim=-1)

            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            total_loss += loss.item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_confs.extend(conf.cpu().numpy())

            for i in range(labels.size(0)):
                c = labels[i].item()
                class_total[c]   += 1
                class_correct[c] += (preds[i] == labels[i]).item()

    accuracy     = 100.0 * correct / total
    avg_loss     = total_loss / len(loader)
    per_class_acc = [100.0 * class_correct[c] / max(class_total[c], 1)
                     for c in range(NUM_CLASSES)]

    return (accuracy, avg_loss,
            per_class_acc,
            np.array(all_preds),
            np.array(all_labels),
            np.array(all_confs))

In [ ]:
# ============================================================
# SECTION 6 — TRAINING LOOP
# Two-stage schedule (mirrors VILLA):
#   Stage 1 (Warm-up epochs 1–2): standard cross-entropy
#   Stage 2 (Adversarial epochs 3–5): VILLA "free" AT
# ============================================================

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# History containers
history = {
    'train_loss_std':  [], 'train_loss_at':  [], 'train_loss_kl':  [],
    'train_loss_total':[], 'train_acc':       [],
    'val_loss':        [], 'val_acc':         [],
    'per_class_acc':   [],   # list of lists (epoch × class)
}

print("\n" + "="*65)
print(" VILLA Training  |  Warm-up: 2 epochs  |  Adversarial: 3 epochs")
print("="*65)

for epoch in range(1, NUM_EPOCHS + 1):

    adversarial_phase = (epoch > WARMUP_EPOCHS)
    phase_label       = "ADV" if adversarial_phase else "STD"
    print(f"\nEpoch [{epoch}/{NUM_EPOCHS}]  Phase: {phase_label}")

    # ---- Training ----
    model.train()
    epoch_std, epoch_at, epoch_kl = 0.0, 0.0, 0.0
    train_correct, train_total    = 0,   0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        if adversarial_phase:
            # VILLA "free" adversarial training
            l_std, l_at, l_kl = free_adversarial_training_step(
                model, images, labels, optimizer,
                eps=EPS, adv_lr=ADV_LR,
                adv_steps=ADV_STEPS, kl_weight=KL_WEIGHT
            )
        else:
            # Standard warm-up training
            l_std, l_at, l_kl = standard_training_step(
                model, images, labels, optimizer
            )

        epoch_std += l_std
        epoch_at  += l_at
        epoch_kl  += l_kl

        # Compute training accuracy (clean forward pass)
        with torch.no_grad():
            logits = model(images)
            preds  = logits.argmax(dim=-1)
            train_correct += (preds == labels).sum().item()
            train_total   += labels.size(0)

        if (batch_idx + 1) % 20 == 0:
            print(f"  Batch [{batch_idx+1}/{len(train_loader)}]"
                  f"  L_std={l_std:.4f}"
                  f"  R_at={l_at:.4f}"
                  f"  R_kl={l_kl:.4f}")

    scheduler.step()

    n_batches     = len(train_loader)
    avg_std       = epoch_std / n_batches
    avg_at        = epoch_at  / n_batches
    avg_kl        = epoch_kl  / n_batches
    avg_total     = avg_std + avg_at + KL_WEIGHT * avg_kl
    train_acc     = 100.0 * train_correct / train_total

    # ---- Validation ----
    val_acc, val_loss, per_cls_acc, _, _, _ = evaluate(model, val_loader)

    # ---- Store history ----
    history['train_loss_std'].append(avg_std)
    history['train_loss_at'].append(avg_at)
    history['train_loss_kl'].append(avg_kl)
    history['train_loss_total'].append(avg_total)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['per_class_acc'].append(per_cls_acc)

    print(f"  → Train Acc: {train_acc:.2f}%  |  Val Acc: {val_acc:.2f}%"
          f"  |  Val Loss: {val_loss:.4f}")

print("\nTraining complete.")

In [ ]:
# ============================================================
# SECTION 7 — FINAL EVALUATION (best epoch)
# ============================================================

best_epoch = int(np.argmax(history['val_acc'])) + 1
best_val_acc = max(history['val_acc'])
print(f"\nBest validation accuracy: {best_val_acc:.2f}%  (epoch {best_epoch})")

# Full evaluation on validation set for confusion matrix & confidences
(final_val_acc, final_val_loss,
 final_per_cls, final_preds,
 final_labels,  final_confs) = evaluate(model, val_loader)

# Separate confidences for correct vs incorrect predictions
correct_mask    = final_preds == final_labels
conf_correct    = final_confs[correct_mask]
conf_incorrect  = final_confs[~correct_mask]
mean_conf_corr  = conf_correct.mean()  if len(conf_correct)   > 0 else 0.0
mean_conf_incorr= conf_incorrect.mean() if len(conf_incorrect) > 0 else 0.0

In [ ]:
# ============================================================
# SECTION 8 — SAMPLE PREDICTIONS VISUALIZATION
# ============================================================

def show_sample_predictions(model, loader, n_samples=8):
    """Display a grid of sample images with predicted vs true labels."""
    model.eval()
    images_shown, preds_shown, labels_shown, confs_shown = [], [], [], []
    # Un-normalise for display
    mean = torch.tensor([0.4914, 0.4822, 0.4465])
    std  = torch.tensor([0.2470, 0.2435, 0.2616])

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs_dev = imgs.to(device)
            logits   = model(imgs_dev)
            probs    = F.softmax(logits, dim=-1)
            conf, pred = probs.max(dim=-1)

            for i in range(imgs.size(0)):
                img_unnorm = imgs[i] * std[:, None, None] + mean[:, None, None]
                img_unnorm = img_unnorm.permute(1, 2, 0).clamp(0, 1).numpy()
                images_shown.append(img_unnorm)
                preds_shown.append(pred[i].item())
                labels_shown.append(lbls[i].item())
                confs_shown.append(conf[i].item())
                if len(images_shown) >= n_samples:
                    break
            if len(images_shown) >= n_samples:
                break

    fig, axes = plt.subplots(2, n_samples // 2, figsize=(14, 5))
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        ax.imshow(images_shown[i])
        ax.axis('off')
        true_name = CIFAR10_CLASSES[labels_shown[i]]
        pred_name = CIFAR10_CLASSES[preds_shown[i]]
        color     = 'green' if preds_shown[i] == labels_shown[i] else 'red'
        ax.set_title(f"T: {true_name}\nP: {pred_name}\n{confs_shown[i]*100:.1f}%",
                     fontsize=8, color=color, fontweight='bold')
    fig.suptitle("VILLA — Sample Predictions (Green=Correct, Red=Wrong)",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('villa_sample_predictions.png', dpi=120, bbox_inches='tight')
    plt.close()
    print("Saved: villa_sample_predictions.png")

show_sample_predictions(model, val_loader, n_samples=8)

In [ ]:
# ============================================================
# SECTION 9 — CONFUSION MATRIX HELPER
# ============================================================

def build_confusion_matrix(true_labels, pred_labels, n_classes):
    """Build and normalise a confusion matrix (row = true, col = pred)."""
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(true_labels, pred_labels):
        cm[t, p] += 1
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    return cm, cm_norm

cm, cm_norm = build_confusion_matrix(final_labels, final_preds, NUM_CLASSES)

In [ ]:
# ============================================================
# SECTION 10 — EPOCH-WISE PER-CLASS ACCURACY HEATMAP DATA
# ============================================================

# per_class_heatmap[epoch, class]
per_class_heatmap = np.array(history['per_class_acc'])   # (5, 10)

In [ ]:
# ============================================================
# SECTION 11 — SUMMARY DASHBOARD  (8-panel, 3-row grid)
# ============================================================

epochs_x = list(range(1, NUM_EPOCHS + 1))

# ---- Colour palette ----
BLUE    = '#2563EB'
RED     = '#DC2626'
GREEN   = '#16A34A'
ORANGE  = '#EA580C'
PURPLE  = '#7C3AED'
TEAL    = '#0891B2'
GREY    = '#6B7280'
YELLOW  = '#CA8A04'
BG      = '#F8FAFC'
DARK    = '#1E293B'

# Custom sequential colour map for heatmap
heat_cmap = LinearSegmentedColormap.from_list(
    'villa', ['#EFF6FF', '#2563EB', '#1E1B4B'])

fig = plt.figure(figsize=(26, 20), facecolor=BG)
fig.suptitle(
    "VILLA — Large-Scale Adversarial Training for V+L  |  CIFAR-10 Experiment Dashboard",
    fontsize=16, fontweight='bold', color=DARK, y=0.98
)

gs = gridspec.GridSpec(
    3, 4,
    figure=fig,
    hspace=0.45, wspace=0.35,
    top=0.94, bottom=0.04, left=0.05, right=0.97
)

ax_A = fig.add_subplot(gs[0, 0])   # A  Total loss curves
ax_B = fig.add_subplot(gs[0, 1])   # B  Component loss curves
ax_C = fig.add_subplot(gs[0, 2])   # C  Accuracy curves
ax_D = fig.add_subplot(gs[0, 3])   # D  Normalised confusion matrix
ax_E = fig.add_subplot(gs[1, 0])   # E  Per-class accuracy bar
ax_F = fig.add_subplot(gs[1, 1:3]) # F  Epoch-wise heatmap (wide)
ax_G = fig.add_subplot(gs[1, 3])   # G  Confidence distribution
ax_H = fig.add_subplot(gs[2, :])   # H  Scorecard (full width)

panel_style = dict(fontsize=11, fontweight='bold', color=DARK, pad=8)

# ── Panel A — Total Loss Curves ───────────────────────────────
ax_A.set_facecolor(BG)
ax_A.plot(epochs_x, history['train_loss_total'], '-o',
          color=BLUE,  linewidth=2.5, markersize=6, label='Train Total Loss')
ax_A.plot(epochs_x, history['val_loss'],         '-s',
          color=RED,   linewidth=2.5, markersize=6, label='Val Loss')
ax_A.axvline(x=WARMUP_EPOCHS + 0.5, color=GREY, linestyle='--',
             linewidth=1.5, alpha=0.7, label='AT activated')
ax_A.fill_between([WARMUP_EPOCHS + 0.5, NUM_EPOCHS + 0.5],
                  0, ax_A.get_ylim()[1] if ax_A.get_ylim()[1] > 0 else 5,
                  alpha=0.06, color=ORANGE)
ax_A.set_title("A — Total Loss Curves", **panel_style)
ax_A.set_xlabel("Epoch", fontsize=9)
ax_A.set_ylabel("Loss", fontsize=9)
ax_A.legend(fontsize=8)
ax_A.set_xticks(epochs_x)
ax_A.grid(True, alpha=0.3, linestyle='--')
ax_A.text(WARMUP_EPOCHS + 0.6, ax_A.get_ylim()[1] * 0.95,
          "ADV\nphase", fontsize=7, color=ORANGE, alpha=0.8)

# ── Panel B — Component Loss Curves (all three VILLA terms) ──
ax_B.set_facecolor(BG)
ax_B.plot(epochs_x, history['train_loss_std'], '-o',
          color=BLUE,   linewidth=2.2, markersize=5, label='$L_{std}$ (Clean CE)')
ax_B.plot(epochs_x, history['train_loss_at'],  '-^',
          color=ORANGE, linewidth=2.2, markersize=5, label='$R_{at}$ (Adv CE)')
ax_B.plot(epochs_x, history['train_loss_kl'],  '-D',
          color=PURPLE, linewidth=2.2, markersize=5, label='$R_{kl}$ (Sym-KL)')
ax_B.axvline(x=WARMUP_EPOCHS + 0.5, color=GREY, linestyle='--',
             linewidth=1.5, alpha=0.7)
ax_B.set_title("B — VILLA Component Losses", **panel_style)
ax_B.set_xlabel("Epoch", fontsize=9)
ax_B.set_ylabel("Loss", fontsize=9)
ax_B.legend(fontsize=8)
ax_B.set_xticks(epochs_x)
ax_B.grid(True, alpha=0.3, linestyle='--')
# Annotate the three loss terms
ax_B.annotate('AT start', xy=(WARMUP_EPOCHS + 0.5, 0),
               xytext=(WARMUP_EPOCHS - 0.3, max(history['train_loss_std']) * 0.5),
               fontsize=7, color=GREY,
               arrowprops=dict(arrowstyle='->', color=GREY, lw=1.0))

# ── Panel C — Accuracy Curves ─────────────────────────────────
ax_C.set_facecolor(BG)
ax_C.plot(epochs_x, history['train_acc'], '-o',
          color=GREEN, linewidth=2.5, markersize=6, label='Train Acc')
ax_C.plot(epochs_x, history['val_acc'],   '-s',
          color=RED,   linewidth=2.5, markersize=6, label='Val Acc')
ax_C.axvline(x=WARMUP_EPOCHS + 0.5, color=GREY, linestyle='--',
             linewidth=1.5, alpha=0.7, label='AT activated')
ax_C.set_title("C — Accuracy Curves", **panel_style)
ax_C.set_xlabel("Epoch", fontsize=9)
ax_C.set_ylabel("Accuracy (%)", fontsize=9)
ax_C.legend(fontsize=8)
ax_C.set_xticks(epochs_x)
ax_C.set_ylim(0, 105)
ax_C.grid(True, alpha=0.3, linestyle='--')
# Mark best val acc
best_idx = int(np.argmax(history['val_acc']))
ax_C.annotate(f"Best\n{best_val_acc:.1f}%",
              xy=(best_epoch, best_val_acc),
              xytext=(best_epoch - 0.8, best_val_acc - 15),
              fontsize=8, color=RED,
              arrowprops=dict(arrowstyle='->', color=RED, lw=1.2))

# ── Panel D — Normalised Confusion Matrix ─────────────────────
im = ax_D.imshow(cm_norm, interpolation='nearest',
                 cmap=heat_cmap, vmin=0, vmax=1)
plt.colorbar(im, ax=ax_D, fraction=0.046, pad=0.04)
tick_marks = np.arange(NUM_CLASSES)
short_names = ['plane','auto','bird','cat','deer',
               'dog','frog','horse','ship','truck']
ax_D.set_xticks(tick_marks)
ax_D.set_yticks(tick_marks)
ax_D.set_xticklabels(short_names, rotation=45, ha='right', fontsize=7)
ax_D.set_yticklabels(short_names, fontsize=7)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        v = cm_norm[i, j]
        ax_D.text(j, i, f"{v:.2f}", ha='center', va='center',
                  fontsize=5.5,
                  color='white' if v > 0.55 else DARK)
ax_D.set_title("D — Normalised Confusion Matrix", **panel_style)
ax_D.set_xlabel("Predicted", fontsize=9)
ax_D.set_ylabel("True", fontsize=9)

# ── Panel E — Per-Class Accuracy Bar Chart ────────────────────
colors_bar = [GREEN if a >= 60 else ORANGE if a >= 40 else RED
              for a in final_per_cls]
bars = ax_E.barh(short_names, final_per_cls,
                 color=colors_bar, edgecolor='white', height=0.65)
ax_E.set_xlim(0, 110)
for bar, acc in zip(bars, final_per_cls):
    ax_E.text(acc + 1, bar.get_y() + bar.get_height() / 2,
              f"{acc:.1f}%", va='center', fontsize=8, color=DARK)
ax_E.axvline(x=np.mean(final_per_cls), color=BLUE,
             linestyle='--', linewidth=1.5, alpha=0.8, label='Mean')
ax_E.set_title("E — Per-Class Accuracy", **panel_style)
ax_E.set_xlabel("Accuracy (%)", fontsize=9)
ax_E.legend(fontsize=8)
ax_E.grid(True, axis='x', alpha=0.3, linestyle='--')

# ── Panel F — Epoch-wise Per-Class Heatmap ────────────────────
im2 = ax_F.imshow(per_class_heatmap.T,
                  aspect='auto', interpolation='nearest',
                  cmap=heat_cmap, vmin=0, vmax=100)
plt.colorbar(im2, ax=ax_F, fraction=0.02, pad=0.02, label='Accuracy (%)')
ax_F.set_xticks(range(NUM_EPOCHS))
ax_F.set_xticklabels([f"E{e}" for e in range(1, NUM_EPOCHS + 1)], fontsize=9)
ax_F.set_yticks(range(NUM_CLASSES))
ax_F.set_yticklabels(CIFAR10_CLASSES, fontsize=8)
for e in range(NUM_EPOCHS):
    for c in range(NUM_CLASSES):
        v = per_class_heatmap[e, c]
        ax_F.text(e, c, f"{v:.0f}",
                  ha='center', va='center', fontsize=7,
                  color='white' if v > 60 else DARK)
# Mark warm-up boundary
ax_F.axvline(x=WARMUP_EPOCHS - 0.5, color=YELLOW,
             linewidth=2.5, linestyle='--', alpha=0.9, label='AT start')
ax_F.legend(fontsize=8, loc='upper left')
ax_F.set_title("F — Epoch-wise Per-Class Accuracy Heatmap", **panel_style)
ax_F.set_xlabel("Epoch", fontsize=9)
ax_F.set_ylabel("Class", fontsize=9)

# ── Panel G — Confidence Distribution Histogram ───────────────
ax_G.set_facecolor(BG)
bins = np.linspace(0, 1, 25)
ax_G.hist(conf_correct,   bins=bins, alpha=0.75,
          color=GREEN, label=f'Correct (μ={mean_conf_corr:.2f})',
          edgecolor='white')
ax_G.hist(conf_incorrect, bins=bins, alpha=0.75,
          color=RED,   label=f'Incorrect (μ={mean_conf_incorr:.2f})',
          edgecolor='white')
ax_G.axvline(mean_conf_corr,   color=GREEN, linewidth=2,
             linestyle='--', alpha=0.9)
ax_G.axvline(mean_conf_incorr, color=RED,   linewidth=2,
             linestyle='--', alpha=0.9)
ax_G.set_title("G — Confidence Distribution", **panel_style)
ax_G.set_xlabel("Predicted Confidence", fontsize=9)
ax_G.set_ylabel("Count", fontsize=9)
ax_G.legend(fontsize=8)
ax_G.grid(True, alpha=0.3, linestyle='--')

# ── Panel H — Summary Scorecard ───────────────────────────────
ax_H.axis('off')
ax_H.set_facecolor('#EFF6FF')

# Compute top / bottom classes
sorted_cls   = sorted(range(NUM_CLASSES), key=lambda c: final_per_cls[c])
bottom_2     = [CIFAR10_CLASSES[c] for c in sorted_cls[:2]]
top_2        = [CIFAR10_CLASSES[c] for c in sorted_cls[-2:]]

scorecard_lines = [
    ("VILLA EXPERIMENT SUMMARY SCORECARD", None, 13, 'bold', DARK),
    ("─" * 115, None, 8, 'normal', GREY),
    (None, None, 0, 'normal', DARK),   # spacer

    # Row 1 — performance
    ("PERFORMANCE METRICS", None, 10, 'bold', BLUE),
    (f"  Best Validation Accuracy   :  {best_val_acc:.2f}%   (Epoch {best_epoch})",
     f"  Final Validation Accuracy  :  {final_val_acc:.2f}%", 9, 'normal', DARK),
    (f"  Final Train Accuracy       :  {history['train_acc'][-1]:.2f}%",
     f"  Total Trainable Params     :  {total_params:,}", 9, 'normal', DARK),

    # Row 2 — loss values
    ("LOSS VALUES (Final Epoch)", None, 10, 'bold', BLUE),
    (f"  L_std  (Clean CE)          :  {history['train_loss_std'][-1]:.4f}",
     f"  R_at   (Adversarial CE)    :  {history['train_loss_at'][-1]:.4f}", 9, 'normal', DARK),
    (f"  R_kl   (Sym-KL Div)        :  {history['train_loss_kl'][-1]:.4f}  "
     f"[weight α = {KL_WEIGHT}]",
     f"  Total Loss (Train)         :  {history['train_loss_total'][-1]:.4f}", 9, 'normal', DARK),

    # Row 3 — class performance
    ("CLASS PERFORMANCE", None, 10, 'bold', BLUE),
    (f"  Top Classes     : {', '.join(top_2)}",
     f"  Bottom Classes  : {', '.join(bottom_2)}", 9, 'normal', DARK),

    # Row 4 — confidence
    ("CONFIDENCE ANALYSIS", None, 10, 'bold', BLUE),
    (f"  Mean Confidence (Correct)    :  {mean_conf_corr*100:.2f}%",
     f"  Mean Confidence (Incorrect)  :  {mean_conf_incorr*100:.2f}%", 9, 'normal', DARK),

    # Row 5 — training setup
    ("TRAINING CONFIGURATION", None, 10, 'bold', BLUE),
    (f"  Warm-up Epochs (Standard AT) :  {WARMUP_EPOCHS}  |  "
     f"Adversarial Epochs  :  {NUM_EPOCHS - WARMUP_EPOCHS}",
     f"  PGD Steps (K)                :  {ADV_STEPS}   |  "
     f"Perturbation ε       :  {EPS}", 9, 'normal', DARK),
    (f"  Adversarial LR (α_adv)       :  {ADV_LR}   |  "
     f"KL Weight (α)        :  {KL_WEIGHT}",
     f"  Batch Size                   :  {BATCH_SIZE}  |  "
     f"Optimizer            :  AdamW (lr=3e-4)", 9, 'normal', DARK),

    ("─" * 115, None, 8, 'normal', GREY),
    ("Perturbation Space: Embedding Layer  |  "
     "Strategy: Free Adversarial Training  |  "
     "Regularisation: Symmetric KL Divergence  |  "
     "Dataset: CIFAR-10 (5 000 train / 1 000 val)",
     None, 8, 'italic', GREY),
]

y_pos = 0.97
for line in scorecard_lines:
    left, right, fsize, fweight, color = line
    if left is None:          # spacer
        y_pos -= 0.018
        continue
    if right is None:
        ax_H.text(0.01, y_pos, left,
                  transform=ax_H.transAxes,
                  fontsize=fsize, fontweight=fweight,
                  color=color, va='top', family='monospace')
    else:
        ax_H.text(0.01, y_pos, left,
                  transform=ax_H.transAxes,
                  fontsize=fsize, fontweight=fweight,
                  color=color, va='top', family='monospace')
        ax_H.text(0.52, y_pos, right,
                  transform=ax_H.transAxes,
                  fontsize=fsize, fontweight=fweight,
                  color=color, va='top', family='monospace')
    y_pos -= (fsize * 0.016 + 0.012)

ax_H.set_title("H — Summary Scorecard", **panel_style)
ax_H.set_facecolor('#EFF6FF')

# ── Save dashboard ────────────────────────────────────────────
dashboard_path = 'villa_cifar10_dashboard.png'
plt.savefig(dashboard_path, dpi=150, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.close()
print(f"\nDashboard saved: {dashboard_path}")
print("All outputs saved successfully.")

# VILLA — Structured Problem–Limitation–Solution Analysis

---

| # | Key Problem / Research Gap | How It Limits Prior Work | Proposed Solution in VILLA |
|---|---|---|---|
| 1 | **No adversarial training framework exists for V+L representation learning** | Prior V+L models (ViLBERT, LXMERT, UNITER) rely solely on standard empirical risk minimization, leaving model generalization entirely dependent on data volume and scale, with no mechanism to explicitly regularize the decision boundary | Introduce VILLA — the first large-scale adversarial training framework designed specifically for multimodal V+L models, applicable to both pre-training and fine-tuning stages |
| 2 | **Overfitting during fine-tuning of large pre-trained V+L models** | Large pre-trained models have enormous parameter capacity, yet downstream task datasets are comparatively small; aggressive fine-tuning consistently leads to overfitting and poor generalization, limiting real-world applicability | Apply adversarial fine-tuning (AFT) as an explicit regularization mechanism during the fine-tuning stage, using task-specific adversarial perturbations to smooth the loss landscape and reduce overfitting |
| 3 | **Adversarial perturbations conventionally applied at the pixel or discrete token level** | Pixel-level image perturbations are rigid, computationally expensive to back-propagate through CNN backbones, and may not transfer meaningfully to feature-level representations; discrete token perturbations are ill-defined since tokens are non-differentiable | Perform adversarial perturbations directly in the **embedding space** — on extracted image-region features and word embeddings — enabling continuous, differentiable, and modality-appropriate perturbation generation |
| 4 | **Adversarial training in the image domain degrades accuracy on clean inputs** | Conventional image-domain AT introduces a well-documented robustness–accuracy trade-off, where gains in adversarial robustness come at the cost of reduced performance on clean data, making AT unsuitable for improving standard benchmark results | Bypass the pixel space entirely by perturbing **pre-extracted region-level feature vectors**, which empirically yields large performance gains on clean inputs without sacrificing accuracy — effectively decoupling robustness from clean-data performance in the multimodal setting |
| 5 | **Standard adversarial training (PGD) is computationally prohibitive at large scale** | $K$-step PGD requires $K$ additional forward-backward passes per update, making it $K$ times more expensive than standard training; this renders large-scale multimodal pre-training with AT practically infeasible | Adopt the **"free" adversarial training strategy**, which accumulates parameter gradients $\nabla_\theta \mathcal{L}$ as a byproduct of computing perturbation gradients $\nabla_\delta \mathcal{L}$ at each PGD step, then performs a single parameter update — achieving the effect of a $K$-times-larger virtual mini-batch at negligible extra cost |
| 6 | **Label-preserving AT constrains only the predicted class, not the full output distribution** | Optimizing cross-entropy loss on adversarial examples only ensures the argmax prediction is preserved; the confidence profile (relative probabilities across all classes) can shift arbitrarily, reducing the smoothness and stability of learned representations | Introduce **KL-divergence regularization** $R_{\text{kl}}(\theta) = \text{KL}(p \| q) + \text{KL}(q \| p)$ that enforces consistency between clean and adversarial output distributions, constraining the full probability vector and promoting a smoother, more stable embedding space |
| 7 | **Adversarial pre-training has not been explored for V+L models** | All prior V+L pre-training methods use only standard objectives (MLM, ITM, MRM); without adversarial regularization during pre-training, the generalization benefits must be introduced separately at fine-tuning, yielding suboptimal transfer across diverse downstream tasks | Introduce **task-agnostic adversarial pre-training (APT)** applied to MLM and ITM objectives, so that improved generalization is baked into the pre-trained representations and transfers uniformly to all downstream tasks before any task-specific supervision is seen |
| 8 | **Adversarial training frameworks lack demonstrated generalizability across V+L architectures** | Prior NLP adversarial methods (FreeLB, SMART) are designed for single-modality Transformer models; their applicability to architecturally diverse multimodal models (one-stream vs. two-stream) is unverified and non-trivial | Validate VILLA on both **UNITER** (single-stream) and **LXMERT** (dual-stream) architectures across six downstream tasks, demonstrating that the framework is model-agnostic and generalizes across fundamentally different multimodal design paradigms |
| 9 | **No principled evaluation of robustness to linguistic variation in V+L models** | Standard V+L benchmarks use fixed question or caption phrasings; models that overfit to surface form rather than underlying semantics are not penalized, making benchmark scores an unreliable proxy for true language understanding | Evaluate on **VQA-Rephrasings**, a dataset of paraphrased VQA questions, as a proxy robustness benchmark; VILLA consistently outperforms UNITER on both original and rephrased splits, demonstrating improved sensitivity to semantic content over surface form |
| 10 | **Lack of interpretability analysis for adversarially trained multimodal models** | Prior V+L work does not systematically analyze whether adversarial training improves the quality of cross-modal attention alignment, leaving the internal representational benefits of AT poorly understood | Conduct **attention-head probing analysis** on visual coreference (Flickr30k Entities) and visual relation (Visual Genome) tasks, quantitatively showing that VILLA learns stronger and more accurate region-phrase and region-region multimodal alignments (0.223 vs. 0.195 average probing score) |

---

## Summary of Thematic Clusters

| Theme | Problems Addressed |
|---|---|
| Training methodology gap | 1, 7 |
| Overfitting and generalization | 2, 6 |
| Modality-appropriate perturbation design | 3, 4 |
| Computational scalability | 5 |
| Architectural generalizability | 8 |
| Robustness and interpretability | 9, 10 |

# VILLA — Related Work Reference Table

---

## 1. Multimodal Pre-training

| Author(s) | Year | Title | Venue | Connection to This Paper |
|---|---|---|---|---|
| Lu et al. | 2019 | ViLBERT: Pretraining Task-Agnostic Visiolinguistic Representations for Vision-and-Language Tasks | NeurIPS | Pioneering two-stream V+L pre-training model; serves as a key baseline and contextual reference for the multimodal pre-training paradigm that VILLA extends with adversarial training |
| Tan & Bansal | 2019 | LXMERT: Learning Cross-Modality Encoder Representations from Transformers | EMNLP | Two-stream V+L model used as a secondary backbone in VILLA generalizability experiments; VILLA-fine is applied to LXMERT on VQA, GQA, and NLVR2 |
| Su et al. | 2019 | VL-BERT: Pre-training of Generic Visual-Linguistic Representations | arXiv | Single-stream V+L pre-training model; cited as a representative of the single-stream design paradigm and used as a comparison baseline in downstream task evaluation |
| Li et al. (Liunian) | 2019 | VisualBERT: A Simple and Performant Baseline for Vision and Language | arXiv | Early single-stream V+L model; included as a baseline in downstream task comparisons across VQA, VCR, and NLVR2 benchmarks |
| Chen et al. | 2019 | UNITER: Learning Universal Image-Text Representations | arXiv | Primary backbone model for VILLA; VILLA is applied to UNITER-base and UNITER-large in both pre-training and fine-tuning stages across all six downstream tasks |
| Li et al. (Gen) | 2019 | Unicoder-VL: A Universal Encoder for Vision and Language by Cross-Modal Pre-training | arXiv | Single-stream V+L pre-training model; cited as a related single-stream approach and included as a comparison baseline in downstream evaluation |
| Li et al. (Xiujun) | 2020 | OSCAR: Object-Semantics Aligned Pre-training for Vision-Language Tasks | arXiv | Recent V+L pre-training model leveraging object tags as anchor points; included as a state-of-the-art comparison baseline in VQA and NLVR2 evaluations |
| Lu et al. | 2019 | 12-in-1: Multi-task Vision and Language Representation Learning | arXiv | Multi-task V+L learning approach; cited as a related method leveraging task co-training to enhance fine-tuning, complementary to VILLA's adversarial regularization strategy |
| Huang et al. | 2020 | Pixel-BERT: Aligning Image Pixels with Text by Deep Multi-modal Transformers | arXiv | Proposes pixel-level image-text alignment as an alternative to region features; contrasts with VILLA's approach of perturbing region-level embeddings rather than raw pixels |
| Alberti et al. | 2019 | Fusion of Detected Objects in Text for Visual Question Answering (B2T2) | arXiv | Single-stream V+L model fusing detected objects into text; cited as part of the broader single-stream model family contextualizing VILLA's architectural choices |

---

## 2. Vision-and-Language Understanding Tasks

| Author(s) | Year | Title | Venue | Connection to This Paper |
|---|---|---|---|---|
| Antol et al. | 2015 | VQA: Visual Question Answering | ICCV | Introduced the VQA task; one of the six downstream tasks on which VILLA is evaluated |
| Goyal et al. | 2017 | Making the V in VQA Matter: Elevating the Role of Image Understanding in Visual Question Answering | CVPR | Introduced VQA v2 benchmark (balanced dataset); the primary VQA evaluation benchmark used for VILLA |
| Zellers et al. | 2019 | From Recognition to Cognition: Visual Commonsense Reasoning | CVPR | Introduced the VCR benchmark requiring commonsense reasoning; one of the six downstream tasks evaluated in VILLA, showing the largest performance gain (+2.9 on Q→AR) |
| Suhr et al. | 2018 | A Corpus for Reasoning about Natural Language Grounded in Photographs (NLVR2) | arXiv | Introduced the NLVR2 benchmark for structured visual reasoning; one of the six downstream tasks evaluated in VILLA |
| Xie et al. | 2019 | Visual Entailment: A Novel Task for Fine-Grained Image Understanding | arXiv | Introduced the Visual Entailment task (SNLI-VE); one of the six downstream tasks on which VILLA is evaluated |
| Yu et al. | 2016 | Modeling Context in Referring Expressions | ECCV | Introduced RefCOCO, RefCOCO+, and RefCOCOg datasets for Referring Expression Comprehension; used as downstream evaluation benchmarks in VILLA |
| Hudson & Manning | 2019 | GQA: A New Dataset for Real-World Visual Reasoning and Compositional Question Answering | CVPR | Introduced the GQA benchmark; used in LXMERT generalizability experiments with VILLA-fine |
| Lee et al. | 2018 | Stacked Cross Attention for Image-Text Matching | ECCV | Introduced image-text retrieval methodology; Flickr30k retrieval task setup follows this work and is one of VILLA's six evaluation benchmarks |
| Shah et al. | 2019 | Cycle-Consistency for Robust Visual Question Answering | CVPR | Introduced the VQA-Rephrasings dataset for robustness evaluation; used in VILLA to test model robustness to linguistic paraphrasing |

---

## 3. Adversarial Training — Core Methods

| Author(s) | Year | Title | Venue | Connection to This Paper |
|---|---|---|---|---|
| Szegedy et al. | 2013 | Intriguing Properties of Neural Networks | arXiv | Foundational work identifying the existence of adversarial examples in neural networks; establishes the theoretical motivation for adversarial training adopted in VILLA |
| Goodfellow et al. | 2014 | Explaining and Harnessing Adversarial Examples | arXiv | Introduced FGSM (Fast Gradient Sign Method); foundational adversarial attack method underlying the perturbation strategy in VILLA |
| Madry et al. | 2017 | Towards Deep Learning Models Resistant to Adversarial Attacks | arXiv | Introduced PGD-based adversarial training; VILLA directly adopts PGD as the inner maximization solver for generating adversarial perturbations |
| Shafahi et al. | 2019 | Adversarial Training for Free! | NeurIPS | Introduced the "free" adversarial training strategy that VILLA adopts to accumulate parameter gradients during PGD steps, enabling efficient large-scale multimodal AT |
| Zhang et al. (Dinghuai) | 2019 | You Only Propagate Once: Accelerating Adversarial Training via Maximal Principle | NeurIPS | Related efficient adversarial training method; cited alongside FreeLB and Free AT as part of the family of computationally efficient AT strategies |
| Zhang et al. (Hongyang) | 2019 | Theoretically Principled Trade-off Between Robustness and Accuracy (TRADES) | arXiv | Introduced KL-divergence-based regularization for adversarial training; directly motivates VILLA's $R_{\text{kl}}$ term enforcing distribution-level consistency between clean and adversarial outputs |
| Miyato et al. | 2018 | Virtual Adversarial Training: A Regularization Method for Supervised and Semi-Supervised Learning | PAMI | Introduced virtual adversarial training using KL divergence for consistency regularization; motivates VILLA's KL regularization component and its use as a generalization-boosting mechanism |

---

## 4. Adversarial Training for Language and Multimodal Models

| Author(s) | Year | Title | Venue | Connection to This Paper |
|---|---|---|---|---|
| Miyato et al. | 2016 | Adversarial Training Methods for Semi-Supervised Text Classification | arXiv | Early work applying adversarial perturbations to word embeddings for NLP; directly motivates VILLA's text-modality perturbation strategy in the embedding space |
| Zhu et al. | 2019 | FreeLB: Enhanced Adversarial Training for Language Understanding | arXiv | Introduced FreeLB, an adversarial training method for NLP using accumulated gradient updates; VILLA adopts and extends this strategy to the multimodal setting and serves as the primary NLP AT baseline for comparison |
| Jiang et al. | 2019 | SMART: Robust and Efficient Fine-Tuning for Pre-trained Natural Language Models | arXiv | Proposed smoothness-inducing adversarial regularization for NLP fine-tuning; motivates VILLA's use of AT as a regularization strategy to combat overfitting in large pre-trained models |
| Chen et al. | 2020 | Adversarial Robustness: From Self-Supervised Pre-training to Fine-Tuning | arXiv | Parallel work exploring adversarial robustness in pre-training; cited as concurrent work that also investigates AT during the pre-training stage |
| Liu et al. | 2020 | Adversarial Training for Large Neural Language Models | arXiv | Recent concurrent work on AT for large-scale language model pre-training; cited as parallel exploration of AT in the pre-training stage for unimodal NLP models |
| Wang et al. | 2019 | Improving Neural Language Modeling via Adversarial Training | arXiv | Applied AT to language modeling; cited as evidence that AT benefits extend to pre-training-stage language objectives, motivating VILLA's adversarial pre-training design |
| Hendrycks et al. | 2019 | Using Pre-Training Can Improve Model Robustness and Uncertainty | arXiv | Demonstrated that pre-training improves robustness of downstream models; supports VILLA's hypothesis that adversarial pre-training generalizes beneficially to fine-tuning |
| Xie et al. (Cihang, 2019) | 2019 | Adversarial Examples Improve Image Recognition | arXiv | Showed that adversarial examples can improve clean-image accuracy when a separate auxiliary batch norm is used; motivates VILLA's finding that embedding-space perturbations improve clean-data performance without accuracy degradation |

---

## 5. Adversarial Attacks in Vision-and-Language

| Author(s) | Year | Title | Venue | Connection to This Paper |
|---|---|---|---|---|
| Chen et al. | 2017 | Attacking Visual Language Grounding with Adversarial Examples: A Case Study on Neural Image Captioning | arXiv | Early work crafting adversarial examples for image captioning; cited to contrast with VILLA's goal — VILLA uses AT to improve model performance rather than to craft interpretable adversarial examples |
| Xu et al. | 2019 | Exact Adversarial Attack to Image Captioning via Structured Output Learning with Latent Variables | CVPR | Studied adversarial attacks targeting image captioning systems; cited as related prior work on V+L adversarial examples, contrasting with VILLA's constructive use of AT |
| Ribeiro et al. | 2018 | Semantically Equivalent Adversarial Rules for Debugging NLP Models | ACL | Investigated textual adversarial rules for attacking VQA; contextualizes the V+L adversarial attack landscape that VILLA navigates differently by focusing on defense and generalization |
| Ramakrishnan et al. | 2018 | Overcoming Language Priors in Visual Question Answering with Adversarial Regularization | NeurIPS | Proposed adversarial regularization to reduce language bias in VQA; explicitly distinguished from VILLA — that work targets language prior removal, not general generalization improvement |

---

## 6. Image-Domain Adversarial Robustness

| Author(s) | Year | Title | Venue | Connection to This Paper |
|---|---|---|---|---|
| Xie et al. (Cihang, 2019) | 2019 | Feature Denoising for Improving Adversarial Robustness | CVPR | Demonstrated the robustness–accuracy trade-off in image classification AT; motivates VILLA's design choice to perturb feature embeddings rather than pixels to avoid this trade-off |
| Tramèr et al. | 2017 | Ensemble Adversarial Training: Attacks and Defenses | arXiv | Explored ensemble-based adversarial defense strategies; cited as part of the broader AT defense literature that informs VILLA's robustness framework |

---

## 7. Pre-training Objectives and Datasets

| Author(s) | Year | Title | Venue | Connection to This Paper |
|---|---|---|---|---|
| Devlin et al. | 2019 | BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding | NAACL | Introduced BERT and the masked language modeling (MLM) pre-training objective; VILLA applies adversarial training to the MLM objective during V+L adversarial pre-training |
| Vaswani et al. | 2017 | Attention is All You Need | NeurIPS | Introduced the Transformer architecture; the core architectural backbone of all V+L models enhanced by VILLA |
| Anderson et al. | 2018 | Bottom-Up and Top-Down Attention for Image Captioning and Visual Question Answering | CVPR | Introduced bottom-up region features from Faster R-CNN; VILLA applies adversarial perturbations directly to these extracted region features as its image-modality input |
| Lin et al. | 2014 | Microsoft COCO: Common Objects in Context | ECCV | Introduced the COCO dataset; one of four large-scale pre-training datasets used in VILLA's adversarial pre-training stage |
| Krishna et al. | 2017 | Visual Genome: Connecting Language and Vision Using Crowdsourced Dense Image Annotations | IJCV | Introduced the Visual Genome dataset; used in VILLA pre-training and in the attention-head probing analysis for visual relation evaluation |
| Sharma et al. | 2018 | Conceptual Captions: A Cleaned, Hypernymed, Image Alt-Text Dataset for Automatic Image Captioning | ACL | Introduced the Conceptual Captions dataset; one of four large-scale pre-training datasets used in VILLA's adversarial pre-training stage |
| Ordonez et al. | 2011 | Im2Text: Describing Images Using 1 Million Captioned Photographs (SBU Captions) | NeurIPS | Introduced SBU Captions dataset; one of four large-scale pre-training datasets used in VILLA's adversarial pre-training stage |
| Plummer et al. | 2015 | Flickr30k Entities: Collecting Region-to-Phrase Correspondences for Richer Image-to-Sentence Models | ICCV | Introduced Flickr30k Entities with region-phrase annotations; used in VILLA's probing analysis to evaluate visual coreference alignment quality |

---

## 8. Consistency and Smoothness Regularization

| Author(s) | Year | Title | Venue | Connection to This Paper |
|---|---|---|---|---|
| Xie et al. (Qizhe) | 2019 | Unsupervised Data Augmentation for Consistency Training (UDA) | arXiv | Proposed consistency-based regularization using KL divergence in semi-supervised learning; motivates VILLA's KL regularization term $R_{\text{kl}}$ that enforces output distribution consistency under perturbation |

---

## 9. Probing and Interpretability of Pre-trained V+L Models

| Author(s) | Year | Title | Venue | Connection to This Paper |
|---|---|---|---|---|
| Cao et al. | 2020 | Behind the Scene: Revealing the Secrets of Pre-trained Vision-and-Language Models | arXiv | Introduced the probing analysis methodology for V+L pre-trained models; VILLA directly adopts this framework to evaluate attention-head quality for visual coreference and visual relation tasks |